In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import preprocessing, features

DATA_DIR = '../datasets/rossmann-store-sales'
STORE_FILE = os.path.join(DATA_DIR, 'store.csv')
TRAIN_FILE = os.path.join(DATA_DIR, 'train.csv')
TEST_FILE = os.path.join(DATA_DIR, 'test.csv')
FORECAST_HORIZON = 6*7 # We're forecasting daily for 6 weeks into the future

from pandas.tseries.offsets import DateOffset

def make_targets(df: pd.DataFrame, horizon: int) -> pd.DataFrame:
    """
    Generate a target DataFrame containing sales shifted consecutive days ahead up to a
    given forecasting horizon.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame containing historical sales data.
        Expected to have date `Date` (Datetime), store ID `Store` (str), and `Sales` columns.
    days_ahead : int
        Number of days ahead to shift the sales data (e.g., 1 for next-day sales).

    Returns
    -------
    pd.DataFrame
        A transformed DataFrame containing:
        - 'Store' and 'Date' as index columns.
        - A single target column named `Sales_ahead_{days_ahead}` containing
          the shifted sales values.
    """

    def make_target(sales_df: pd.DataFrame, days_ahead: int) -> pd.DataFrame:
        """ Generate a target DataFrame containing sales shifted by a specified number of days ahead. """

        targets = (sales_df
                .shift(freq=DateOffset(days=-days_ahead))
                .iloc[days_ahead:]
                .reset_index()
                .melt(id_vars=['Date'], value_name=f'Sales_ahead_{days_ahead}')
                .set_index(['Store', 'Date'])
                )

        return targets

    # Shift sales various times to gather all targets for the forecasting horizon
    sales_df = pd.pivot(df, index='Date', columns='Store', values='Sales')
    targets = [make_target(sales_df, day) for day in range(1, horizon+1)]

    # Merge and filter Store-Date combinations that do not appear in the training set
    targets = pd.concat(targets, axis=1).reset_index()
    targets = df.merge(targets, on=['Store', 'Date'], how='left').loc[:, targets.columns]

    return targets


In [2]:
store_df = pd.read_csv(STORE_FILE)
store_df = preprocessing.store_data(store_df)

# NOTE: train_df['Open'] == 0 -> train_df['Sales'] = 0. This happens always
train_df = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1) # Not available in test
train_df = features.attach_store_data(train_df, store_df)

test_df = pd.read_csv(TEST_FILE, index_col=0, parse_dates=['Date'])
test_df = features.attach_store_data(test_df, store_df)

C:\Users\m_kal\AppData\Local\Temp\ipykernel_17348\2952918199.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1) # Not available in test


In [3]:
""" Feature engineering """

# Competition-related features
train_df['CompetitionDistance'] = train_df['CompetitionDistance'].apply(np.log1p)
train_df['CompetitionSinceMonths'] = ( (train_df['Date'] - train_df['CompetitionSinceDate']).dt.days / 30.0 ).round()

# Promotion related features
train_df['Promo2SinceWeeks'] =  ( (train_df['Date'] - train_df['Promo2SinceDate']).dt.days / 7.0 ).fillna(0).round().astype(int) * train_df['Promo2']

# Basic date features
train_df['WeekOfYear'] = train_df['Date'].dt.isocalendar().week
train_df['Month'] = train_df['Date'].dt.month
train_df['Year'] = train_df['Date'].dt.year
train_df['Quarter'] = train_df['Date'].dt.quarter

# Calendar and seasonality features
train_df['is_weekend'] = train_df['Date'].dt.dayofweek >= 5

# Cyclical features
train_df['Month_sin'] = np.sin(2 * np.pi * train_df['Month'] / 12)
train_df['Month_cos'] = np.cos(2 * np.pi * train_df['Month'] / 12)
train_df['Dayofweek_sin'] = np.sin(2 * np.pi * train_df['DayOfWeek'] / 7)
train_df['Dayofweek_cos'] = np.cos(2 * np.pi * train_df['DayOfWeek'] / 7)

# Drop useless
#train_df.drop(['Promo2SinceDate', 'CompetitionSinceDate'], axis=1, inplace=True)


In [110]:

# Generate target dataframe
targets = make_targets(train_df[['Store', 'Date', 'Sales']], horizon=FORECAST_HORIZON)

In [ ]:

# TODO: Lag and rolling features - NOTE: We are predicting 6 weeks ahead.
train_df['lag_1'] = train_df['target'].shift(1)
train_df['rolling_mean_7'] = train_df['target'].shift(1).rolling(7).mean()
train_df['rolling_std_7'] = train_df['target'].shift(1).rolling(7).std()


In [107]:
from typing import Iterable

def make_lag_df(df: pd.DataFrame, lag: DateOffset) -> pd.DataFrame:
        
    id_name = df.index.name
    value_name = "_".join(f"lag_{k}_{v}" for k, v in lag.kwds.items())
    melt_index = [df.index.name, df.columns.name]

    lag_df = (df.shift(freq=lag)
              .reset_index()
              .melt(id_vars=[id_name], value_name=value_name)
              .set_index(melt_index)
              )

    return lag_df

def make_lags(df: pd.DataFrame, lags: DateOffset | Iterable[DateOffset]) -> pd.DataFrame:

    if isinstance(lags, DateOffset):
        lags = [lags]

    df_pivot = (pd.pivot(train_df, index=df.columns[0], columns=df.columns[1], values=df.columns[2])
                .sort_index()) # Sorted from oldest to newest

    lag_dfs = [make_lag_df(df_pivot, lag) for lag in lags]

    lag_df = pd.concat(lag_dfs, axis=1).reset_index()
    lag_df_merged = df.merge(lag_df, how='left').loc[:, lag_df.columns]

    return lag_df_merged

In [109]:
df = train_df[['Date', 'Store', 'Sales']] # 1st col is datetime, 2nd col is ID, 3rd is value
lags = [1, 2, 3, 4, 5, 6, 7] # days
offsets = [DateOffset(days=lag) for lag in lags]
lag_df = make_lags(df, offsets)

,Date,Store,lag_days_1,lag_days_2,lag_days_3,lag_days_4,lag_days_5,lag_days_6,lag_days_7
0,2015-07-31,1,5020.0,4782.0,5011.0,6102.0,0.0,4364.0,3706.0
1,2015-07-31,2,5567.0,6402.0,5671.0,6627.0,0.0,2512.0,3854.0
2,2015-07-31,3,8977.0,7610.0,8864.0,8107.0,0.0,3878.0,5080.0
3,2015-07-31,4,10387.0,10514.0,10275.0,11812.0,0.0,9322.0,8322.0
4,2015-07-31,5,4943.0,5899.0,6083.0,7059.0,0.0,2030.0,3815.0
...,...,...,...,...,...,...,...,...,...
1017204,2013-01-01,1111,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1017205,2013-01-01,1112,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1017206,2013-01-01,1113,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1017207,2013-01-01,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [115]:
lags = [1, 2, 3, 4, 5, 6, 7] # days
offsets = [DateOffset(days=lag) for lag in list(range(-1, -FORECAST_HORIZON-1, -1))]
lag_df = make_lags(df, offsets)
lag_df

,Date,Store,lag_days_-1,lag_days_-2,lag_days_-3,lag_days_-4,lag_days_-5,lag_days_-6,lag_days_-7,lag_days_-8,...,lag_days_-33,lag_days_-34,lag_days_-35,lag_days_-36,lag_days_-37,lag_days_-38,lag_days_-39,lag_days_-40,lag_days_-41,lag_days_-42
0,2015-07-31,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-07-31,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-07-31,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-07-31,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-07-31,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1017204,2013-01-01,1111,5097.0,4579.0,4640.0,3325.0,0.0,9444.0,6472.0,5307.0,...,0.0,8441.0,7337.0,4881.0,5967.0,5737.0,3726.0,0.0,2830.0,4891.0
1017205,2013-01-01,1112,10797.0,8716.0,9788.0,9513.0,0.0,25165.0,17058.0,14724.0,...,0.0,23498.0,16412.0,14723.0,13911.0,13539.0,10632.0,0.0,9343.0,8797.0
1017206,2013-01-01,1113,6218.0,5563.0,5524.0,5194.0,0.0,8984.0,6866.0,6115.0,...,0.0,8563.0,6234.0,7132.0,7638.0,6828.0,5357.0,0.0,5001.0,5013.0
1017207,2013-01-01,1114,20642.0,18463.0,18371.0,18856.0,0.0,21237.0,18816.0,17073.0,...,0.0,22125.0,18715.0,19167.0,19514.0,19667.0,19873.0,0.0,16978.0,15364.0


In [111]:
targets

,Store,Date,Sales_ahead_1,Sales_ahead_2,Sales_ahead_3,Sales_ahead_4,Sales_ahead_5,Sales_ahead_6,Sales_ahead_7,Sales_ahead_8,...,Sales_ahead_33,Sales_ahead_34,Sales_ahead_35,Sales_ahead_36,Sales_ahead_37,Sales_ahead_38,Sales_ahead_39,Sales_ahead_40,Sales_ahead_41,Sales_ahead_42
0,1,2015-07-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2015-07-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2015-07-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,2015-07-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2015-07-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1017204,1111,2013-01-01,5097.0,4579.0,4640.0,3325.0,0.0,9444.0,6472.0,5307.0,...,0.0,8441.0,7337.0,4881.0,5967.0,5737.0,3726.0,0.0,2830.0,4891.0
1017205,1112,2013-01-01,10797.0,8716.0,9788.0,9513.0,0.0,25165.0,17058.0,14724.0,...,0.0,23498.0,16412.0,14723.0,13911.0,13539.0,10632.0,0.0,9343.0,8797.0
1017206,1113,2013-01-01,6218.0,5563.0,5524.0,5194.0,0.0,8984.0,6866.0,6115.0,...,0.0,8563.0,6234.0,7132.0,7638.0,6828.0,5357.0,0.0,5001.0,5013.0
1017207,1114,2013-01-01,20642.0,18463.0,18371.0,18856.0,0.0,21237.0,18816.0,17073.0,...,0.0,22125.0,18715.0,19167.0,19514.0,19667.0,19873.0,0.0,16978.0,15364.0
